# Qwen3.8 ARC on Google Colab

This notebook clones the repository and prepares a **development pilot** and a separate **Phase A** workflow. FP8/A100 results are exploratory and distinct from previous BF16 and MLX results. Opening this notebook does not start a runtime or an experiment. Select an A100 GPU in Runtime settings before executing cells. Colab charges compute units while connected.

Enter an exact reviewed commit SHA containing this notebook and its support code. Place `arc-agi_training_challenges.json` in the Drive input directory. The solutions file stays on your local computer. Never upload solutions to this runtime.


In [ ]:
# User configuration: set these values before running setup.
REPO_REVISION = ""  # Required: 40-character reviewed commit containing this notebook.
DRIVE_INPUT = "/content/drive/MyDrive/qwen-colab/inputs"
DRIVE_RUNS = "/content/drive/MyDrive/qwen-colab/runs"
MODEL_ID = "Qwen/Qwen3.8-27B-FP8"
MODEL_REVISION = "c3a462cc3d38cf351f3b26cbd763469eeb74ef81"
VLLM_VERSION = "0.33.0"
MAX_MODEL_LEN = 18432


## 1. Mount Drive and clone the exact reviewed repository
Mounting Drive grants the runtime access to Drive files. Keep held-out solutions outside the mounted Drive account while inference runs; passing only public tasks is not an operating-system isolation boundary.

In [ ]:
import os, re, subprocess, sys
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
from pathlib import Path
assert re.fullmatch(r'[0-9a-f]{40}', REPO_REVISION), 'Set REPO_REVISION to a reviewed commit SHA'
repo = Path('/content/qwen-meetup-2')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/morshoto/qwen-meetup-2.git',str(repo)],check=True)
assert not subprocess.check_output(['git','-C',str(repo),'status','--porcelain'],text=True).strip(), 'Repository has local changes'
subprocess.run(['git','-C',str(repo),'fetch','origin',REPO_REVISION],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',REPO_REVISION],check=True)
assert subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()==REPO_REVISION
os.chdir(repo)
sys.path.insert(0,str(repo))
# A reused kernel may already contain modules from an older checkout.
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'): del sys.modules[name]
print('Repository commit:', REPO_REVISION)
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],text=True))


## 2. Install and verify the runtime
This installs packages in the Colab runtime and downloads around 31 GB of model files. Runtime and GPU availability are not guaranteed.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','vllm=='+VLLM_VERSION,'transformers>=5.8.0','huggingface_hub','cuda-toolkit[nvcc]==13.0.3.0','ninja'],check=True)
import nvidia
cuda_home=Path(next(iter(nvidia.__path__)))/'cu13'
os.environ['CUDA_HOME']=str(cuda_home)
os.environ['PATH']=str(Path(sys.executable).parent)+os.pathsep+str(cuda_home/'bin')+os.pathsep+os.environ['PATH']
os.environ['LD_LIBRARY_PATH']=str(cuda_home/'lib')+os.pathsep+os.environ.get('LD_LIBRARY_PATH','')
for target, link in [('lib',cuda_home/'lib64'),('libcudart.so.13',cuda_home/'lib'/'libcudart.so')]:
    if not link.exists(): link.symlink_to(target)
subprocess.run(['nvcc','--version'],check=True)
subprocess.run([sys.executable,'-m','src.cli.lt_h100_preflight','--vllm-version',VLLM_VERSION],check=True)


In [ ]:
import hashlib, json
from huggingface_hub import snapshot_download
snapshot=Path(snapshot_download(MODEL_ID,revision=MODEL_REVISION))
def digest(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest()
files={p.name:digest(p) for p in sorted(snapshot.iterdir()) if p.is_file()}
assert set(files)=={p.name for p in snapshot.iterdir()}, 'Unexpected model inventory'
model_info={'model_id':MODEL_ID,'revision':MODEL_REVISION,'files_sha256':files,'chat_template_sha256':digest(snapshot/'chat_template.jinja')}
inputs=Path(DRIVE_INPUT); inputs.mkdir(parents=True,exist_ok=True)
(inputs/'model-info.json').write_text(json.dumps(model_info,indent=2)+'\n')
(inputs/'phase-a-model-config.json').write_text(json.dumps({'model':model_info,'backend':{'name':'vllm','version':VLLM_VERSION,'max_model_len':MAX_MODEL_LEN}},indent=2)+'\n')
print('Model inventory and local-registration config saved to', inputs)


## 3. Development-only pilot
Generate public task exports from the challenges file; no solutions are needed. Set a unique run name and an explicit time and estimated compute budget. The model is loaded once per task by the registered worker. A failed or timed-out task is recorded and is not retried automatically.

In [ ]:
import shutil, time
from src.arc.lt_development_export import build_development_export
challenge=inputs/'arc-agi_training_challenges.json'
assert challenge.is_file(), f'Missing {challenge}'
raw_dir=repo/'data/raw'; raw_dir.mkdir(parents=True,exist_ok=True)
shutil.copy2(challenge,raw_dir/challenge.name)
export, receipt=build_development_export(challenge.read_bytes(),(repo/'research/rsi/bounded-exposure-proposal.json').read_bytes(),json.loads((repo/'configs/lt.phase-a.development.json').read_text()))
from transformers import AutoTokenizer
from src.arc.lt_live_backend import PROMPT
tokenizer=AutoTokenizer.from_pretrained(str(snapshot),local_files_only=True)
def preflight_context(tasks):
    sizes={}
    for task_id,task in tasks.items():
        text=json.dumps({'instruction':PROMPT,'task':task},sort_keys=True,separators=(',',':'))
        formatted=tokenizer.apply_chat_template([{'role':'user','content':text}],tokenize=False,add_generation_prompt=True,enable_thinking=False)
        sizes[task_id]=len(tokenizer.encode(formatted,add_special_tokens=False))
    largest=max(sizes.values())
    print('Largest input:',largest,'tokens; registered context:',MAX_MODEL_LEN)
    if largest+4096>MAX_MODEL_LEN: raise ValueError('At least one prompt plus the 4096-token output budget exceeds the registered context')
preflight_context(export['tasks'])


In [ ]:
# Choose a new name for a new registration, or reuse one to resume.
RUN_NAME = ""  # e.g. development-20261108-a
MAX_MINUTES = 0  # required positive number
MAX_COMPUTE_UNITS = 0  # required positive number
UNITS_PER_HOUR = 0  # enter the current Colab rate
assert RUN_NAME and re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*',RUN_NAME)
from src.arc.lt_colab_budget import Budget
budget=Budget(minutes=MAX_MINUTES,units=MAX_COMPUTE_UNITS,rate_per_hour=UNITS_PER_HOUR)
run_dir=Path(DRIVE_RUNS)/RUN_NAME
run_dir.parent.mkdir(parents=True,exist_ok=True)
from src.cli.lt_vllm_pilot import prepare, _manifest
if run_dir.exists():
    expected = _manifest(export, receipt, model_info, VLLM_VERSION, MAX_MODEL_LEN)
    if json.loads((run_dir / 'manifest.json').read_text()) != expected:
        raise ValueError('Existing pilot registration differs; choose a new run name')
    print('Resuming pilot registration:', expected['sha256'])
else:
    print('Pilot registration:', prepare(run_dir, export, receipt, model_info, VLLM_VERSION, MAX_MODEL_LEN))


In [ ]:
# Run this cell explicitly to start or continue the development pilot.
# Estimated budget covers this launch only and is not a provider-enforced cap.
from src.cli.lt_vllm_pilot import run as run_pilot
started=time.monotonic()
for _ in range(3):
    timeout=budget.task_timeout(time.monotonic()-started,1800)
    if timeout is None: print('Budget exhausted before another task'); break
    completed=run_pilot(run_dir,max_tasks=1,timeout_seconds=timeout)
    if not completed: print('No pending tasks'); break
    rows=sorted((run_dir/'single').glob('*.result.json'),key=lambda p:p.stat().st_mtime)
    if rows and json.loads(rows[-1].read_text())['status']!='complete': print('Stopped after failure'); break
print('Elapsed seconds:',round(time.monotonic()-started,2),'estimated units:',round((time.monotonic()-started)*UNITS_PER_HOUR/3600,3))
print('Journal:',run_dir)


## 4. Phase A: separate registration and explicit launch
Do not use the development pilot as a benchmark. On your local machine, check out the **same exact commit**, keep the ARC solutions locally, and prepare the run with `python -m src.cli.lt_single prepare --config phase-a-model-config.json --directory NEW_DIRECTORY`. Copy only its `manifest.json`, `export.json`, and `receipt.json` to a new Drive run directory. Copy the public challenges file to the input directory; never copy the solutions file. The runner verifies the registration and public export before each task; scoring remains local with the solutions file.

Inspect the pilot, choose a new budget and enter `PHASE_A_APPROVAL` exactly as shown only when ready. Do not resume a terminal failure or interrupted task without inspecting its journal and cleanup evidence.

In [ ]:
PHASE_A_RUN = ""  # Drive path containing locally prepared manifest.json/export.json/receipt.json
PHASE_A_APPROVAL = ""  # enter RUN_PHASE_A only after reviewing the registration and pilot
PHASE_A_MINUTES = 0
PHASE_A_UNITS = 0
PHASE_A_RATE = 0


In [ ]:
assert PHASE_A_APPROVAL=='RUN_PHASE_A', 'Phase A launch is not approved'
from src.arc.lt_colab_budget import Budget
phase_budget=Budget(minutes=PHASE_A_MINUTES,units=PHASE_A_UNITS,rate_per_hour=PHASE_A_RATE)
phase_dir=Path(PHASE_A_RUN)
assert phase_dir.is_dir() and (phase_dir/'manifest.json').is_file()
from src.arc.lt_single_manifest import verify
manifest=json.loads((phase_dir/'manifest.json').read_text())
verify(repo,manifest,public_only=True)
assert manifest['model']==model_info and manifest['backend']=={'name':'vllm','version':VLLM_VERSION,'max_model_len':MAX_MODEL_LEN}
phase_export=json.loads((phase_dir/'export.json').read_text())
preflight_context(phase_export['tasks'])
from src.cli.lt_single import run as run_phase_a
started=time.monotonic()
for _ in range(len(manifest['task_ids'])):
    timeout=phase_budget.task_timeout(time.monotonic()-started,manifest['task_timeout_seconds'])
    if timeout is None: print('Budget exhausted before another task'); break
    completed=run_phase_a(phase_dir,max_tasks=1,timeout_seconds=timeout,public_only=True)
    if not completed: print('No pending tasks'); break
    rows=sorted((phase_dir/'single').glob('*.result.json'),key=lambda p:p.stat().st_mtime)
    if rows and json.loads(rows[-1].read_text())['status']!='complete': print('Stopped after failure'); break
print('Elapsed seconds:',round(time.monotonic()-started,2),'estimated units:',round((time.monotonic()-started)*PHASE_A_RATE/3600,3))


### Recover and score
Copy the entire run directory from Drive to the same reviewed repository on your Mac. Use `python -m src.cli.lt_single score --directory PATH` with the locally retained challenges and solutions. Review interrupted tasks before using the existing `reconcile` operation. When finished, disconnect the Colab runtime in the Runtime menu to stop compute-unit usage; backing files on Drive remain.